# Clustering — K-Means y Jerárquico _(versión detallada)_

_Distancias, elección de K, dendrogramas y un caso real de segmentación de clientes_

**Módulo 2 — Aprendizaje No Supervisado** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

---
> 📖 **Nota:** Esta es la versión detallada del notebook `02_clustering_kmeans_jerarquico.ipynb`.  
> Incluye teoría ampliada y comentarios línea a línea en cada bloque de código.

![Aprendizaje No Supervisado](assets/header.png)

## 1. ¿Qué es clustering?

Dado un conjunto de observaciones $\{x_1, \dots, x_n\}$ **sin etiquetas**, queremos partirlas en $K$ grupos (clusters) tales que:

- Los puntos **dentro** de un grupo se parezcan entre sí (**alta cohesión interna**)
- Los puntos de **distintos** grupos sean diferentes (**alta separación entre clusters**)

### Formulación matemática

Buscamos una partición $C_1, \dots, C_K$ que minimice una medida de **disimilitud intra-cluster**:

$$
\min_{C_1,\dots,C_K} \sum_{k=1}^{K} \sum_{x_i \in C_k} d(x_i, \mu_k)
$$

donde:
- $\mu_k$: centro (centroide) del cluster $k$
- $d(\cdot,\cdot)$: métrica de distancia (euclidiana, Manhattan, coseno, etc.)

### Diferencia con clasificación supervisada

| Aspecto | Clasificación (Supervisado) | Clustering (No Supervisado) |
|---------|----------------------------|----------------------------|
| **Etiquetas** | Conocidas de antemano | No existen — las descubrimos |
| **Objetivo** | Predecir clase de nuevos datos | Descubrir grupos naturales |
| **Evaluación** | Objetiva (accuracy, F1, etc.) | Subjetiva (silhouette, validación de negocio) |
| **Número de grupos** | Fijo (dado por los datos) | Hay que elegirlo ($K$) |

### Casos de uso

- **Marketing:** Segmentación de clientes (comportamiento, valor, riesgo de churn)
- **Biología:** Agrupamiento de genes con expresión similar
- **Retail:** Agrupamiento de productos para recomendaciones
- **Salud:** Identificación de subtipos de enfermedades
- **Visión:** Segmentación de imágenes, compresión de colores

> 💡 **Clave:** El clustering no "predice" — **describe** la estructura de los datos para tomar decisiones de negocio.

## 2. Métricas de distancia — el corazón del clustering

La métrica de distancia define qué significa "parecido". Elegir la métrica correcta es **crítico** para el éxito del clustering.

### Propiedades de una métrica

Una función $d: \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ es una métrica si cumple:

1. **No negatividad:** $d(x, y) \ge 0$
2. **Identidad:** $d(x, y) = 0 \iff x = y$
3. **Simetría:** $d(x, y) = d(y, x)$
4. **Desigualdad triangular:** $d(x, z) \le d(x, y) + d(y, z)$

### Métricas más comunes

#### 1. Distancia Euclidiana (L2)
**La más usada.** Distancia "en línea recta" en el espacio euclidiano.

$$
d_{\text{euc}}(x, y) = \sqrt{\sum_{j=1}^{p} (x_j - y_j)^2} = \|x - y\|_2
$$

**Propiedades:**
- Sensible a la **escala** → siempre escalar antes
- Asume todas las dimensiones son igualmente importantes
- K-Means usa euclidiana por defecto

**Cuándo usar:** Variables numéricas continuas, clusters esféricos.

---

#### 2. Distancia de Manhattan (L1)
También llamada **city block** o **taxicab distance**.

$$
d_{\text{man}}(x, y) = \sum_{j=1}^{p} |x_j - y_j| = \|x - y\|_1
$$

**Propiedades:**
- Más **robusta a outliers** que euclidiana (no eleva al cuadrado)
- Útil cuando el movimiento está restringido a ejes (como en una cuadrícula)

**Cuándo usar:** Datos con outliers, variables en escalas muy diferentes.

---

#### 3. Distancia de Minkowski (generalización)
Familia que incluye euclidiana y Manhattan como casos especiales.

$$
d_{\text{mink}}(x, y) = \left(\sum_{j=1}^{p} |x_j - y_j|^q\right)^{1/q}
$$

- $q=1$: Manhattan
- $q=2$: Euclidiana
- $q \to \infty$: Distancia de Chebyshev (máxima diferencia en cualquier dimensión)

---

#### 4. Distancia del Coseno
Mide el **ángulo** entre vectores, ignorando su magnitud.

$$
d_{\cos}(x, y) = 1 - \frac{x \cdot y}{\|x\| \, \|y\|} = 1 - \cos(\theta)
$$

**Propiedades:**
- **Ignora la magnitud** — solo importa la dirección
- Invariante a multiplicación por escalar

**Cuándo usar:** 
- **Text mining:** Comparar documentos (TF-IDF, bag-of-words)
- **Sistemas de recomendación:** Perfiles de usuarios/productos
- **Embeddings:** word2vec, BERT, etc.

---

#### 5. Distancia de Mahalanobis
Toma en cuenta la **covarianza** entre variables.

$$
d_{\text{mah}}(x, y) = \sqrt{(x - y)^\top \Sigma^{-1} (x - y)}
$$

donde $\Sigma$ es la matriz de covarianza.

**Propiedades:**
- Escala automáticamente según la varianza de cada dimensión
- Detecta correlaciones entre variables

**Cuándo usar:** Variables correlacionadas, detección de outliers.

---

> ⚠️ **Importante:** K-Means está **atado a la distancia euclidiana** (los centroides son medias). Para usar otras distancias, considera **K-Medoids** (PAM) o **clustering jerárquico**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# Visualizamos cómo cambia el "vecino más cercano" según la métrica
# Esto es clave para entender por qué la elección de métrica importa

# Generamos 40 puntos aleatorios en 2D
rng = np.random.default_rng(0)
puntos = rng.uniform(0, 10, size=(40, 2))

# Punto de referencia (estrella roja) — desde aquí medimos distancias
ref = np.array([[5, 5]])

# Creamos 3 subplots para comparar métricas
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Probamos 3 métricas diferentes
metricas = ['euclidean', 'cityblock', 'cosine']
titulos = ['Euclidiana (L2)', 'Manhattan (L1)', 'Coseno']

for ax, metric, titulo in zip(axes, metricas, titulos):
    # cdist calcula distancias desde ref a todos los puntos
    # Devuelve una matriz (1 × 40) que aplanamos con .ravel()
    d = cdist(ref, puntos, metric=metric).ravel()
    
    # Scatter plot coloreado por distancia
    # Colores más oscuros = mayor distancia al punto rojo
    sc = ax.scatter(puntos[:, 0], puntos[:, 1], c=d, cmap='viridis', 
                    s=60, edgecolor='white')
    
    # Punto de referencia (estrella roja)
    ax.scatter(*ref[0], marker='*', s=300, c='red', 
               edgecolor='black', label='Referencia')
    
    ax.set_title(titulo)
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.legend()
    
    # Colorbar para interpretar las distancias
    plt.colorbar(sc, ax=ax, label='Distancia')

plt.tight_layout()
plt.show()

# OBSERVACIONES CLAVE:
# - Euclidiana: círculos concéntricos (isotropía — todas las direcciones iguales)
# - Manhattan: diamantes (movimiento restringido a ejes, como en una ciudad)
# - Coseno: depende del ángulo, no de la distancia física
#   Puntos en la misma dirección tienen distancia pequeña aunque estén lejos
#
# IMPLICACIÓN: El mismo conjunto de puntos puede agruparse diferente
# según la métrica elegida. No hay una "correcta" — depende del problema.


## 3. K-Means

### 3.1 Idea central

K-Means encuentra $K$ centroides $\mu_1, \dots, \mu_K$ y asigna cada punto al centroide más cercano. Minimiza la **inercia** (suma de cuadrados intra-cluster):

$$
J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2
$$

**Intuición:** Queremos que los puntos de cada cluster estén lo más cerca posible de su centroide.

### 3.2 Algoritmo de Lloyd (K-Means estándar)

**Entrada:** Datos $X \in \mathbb{R}^{n \times p}$, número de clusters $K$

**Salida:** Asignación de cada punto a un cluster, centroides $\mu_1, \dots, \mu_K$

**Pasos:**

1. **Inicialización:** Elegir $K$ centroides iniciales
   - **Random:** $K$ puntos aleatorios de los datos
   - **k-means++:** Elegir centroides dispersos (mejor convergencia) ← **Recomendado**

2. **Asignación:** Para cada punto $x_i$, asignarlo al cluster del centroide más cercano:
   $$
   c_i = \arg\min_{k} \| x_i - \mu_k \|^2
   $$

3. **Actualización:** Recalcular cada centroide como la **media** de los puntos asignados:
   $$
   \mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i
   $$

4. **Convergencia:** Repetir pasos 2-3 hasta que:
   - Los centroides no cambien (o cambien muy poco)
   - Se alcance el número máximo de iteraciones

**Propiedades del algoritmo:**
- ✅ **Garantiza convergencia** (la inercia nunca aumenta)
- ❌ **No garantiza óptimo global** — puede quedar atrapado en mínimos locales
- ❌ **Sensible a la inicialización** — por eso se usa `n_init` (múltiples inicializaciones)

### 3.3 Inicialización k-means++

Mejora la inicialización aleatoria eligiendo centroides **dispersos**:

1. Elegir el primer centroide $\mu_1$ uniformemente al azar
2. Para $k = 2, \dots, K$:
   - Calcular $D(x_i)$ = distancia de cada punto al centroide más cercano ya elegido
   - Elegir $\mu_k$ con probabilidad proporcional a $D(x_i)^2$
   - (Puntos lejos de centroides existentes tienen más probabilidad)

**Resultado:** Centroides iniciales bien separados → mejor convergencia, menos iteraciones.

### 3.4 Limitaciones importantes

| Limitación | Descripción | Solución |
|------------|-------------|----------|
| **Necesita fijar $K$** | No sabemos cuántos clusters hay | Método del codo, silhouette, gap statistic |
| **Asume clusters esféricos** | Por usar distancia euclidiana | Usar clustering jerárquico, DBSCAN, GMM |
| **Sensible a escala** | Variables con mayor magnitud dominan | **StandardScaler** antes de K-Means |
| **Sensible a outliers** | Un outlier puede "jalar" un centroide | Detectar y remover outliers, o usar K-Medoids |
| **Solo distancia euclidiana** | No puede usar otras métricas | Usar K-Medoids (PAM) o clustering jerárquico |

> 💡 **Regla de oro:** Siempre escalar los datos con `StandardScaler` antes de K-Means.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# Generamos datos sintéticos con 4 clusters bien separados
# n_samples=400: 400 observaciones (100 por cluster)
# centers=4: 4 centroides verdaderos (ground truth)
# cluster_std=0.9: desviación estándar dentro de cada cluster (ruido)
# random_state=0: semilla fija para reproducibilidad
X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=0)

# Aplicamos K-Means con K=4
# n_clusters=4: número de clusters a encontrar
# n_init=10: número de inicializaciones diferentes (elige la mejor por inercia)
#            Más alto → más robusto pero más lento
# random_state=0: semilla para reproducibilidad
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)

# --- Visualización ---
fig, ax = plt.subplots(figsize=(7, 5))

# Scatter plot de los puntos coloreados por cluster asignado
# km.labels_: array con el cluster asignado a cada punto (0, 1, 2, 3)
# cmap='tab10': paleta de colores categóricos
ax.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap='tab10', s=25, alpha=0.8)

# Centroides finales (marcados con X negra)
# km.cluster_centers_: array (4 × 2) con las coordenadas de los 4 centroides
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
           marker='X', c='black', s=200, edgecolor='white', 
           linewidth=2, label='Centroides')

ax.set_title('K-Means con K=4 sobre datos sintéticos')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Métricas del modelo
print(f'Inercia (suma de distancias al centroide²): {km.inertia_:.2f}')
print(f'Número de iteraciones hasta convergencia: {km.n_iter_}')

# OBSERVACIONES:
# - Los 4 clusters están bien separados (colores distintos)
# - Los centroides (X negras) están en el "centro de masa" de cada grupo
# - La inercia mide qué tan compactos son los clusters (menor = mejor)
# - En este caso, K-Means recupera perfectamente los 4 grupos verdaderos


## 4. ¿Cómo elegir el K óptimo?

Esta es **la pregunta más difícil** en clustering. No hay una respuesta universal — depende del problema y del objetivo de negocio.

### 4.1 Método del codo (Elbow Method)

**Idea:** Graficar la **inercia** $J(K)$ vs $K$ y buscar el "codo" — el punto donde la mejora marginal se aplana.

**Fórmula de la inercia:**

$$
J(K) = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2
$$

**Propiedades:**
- La inercia **siempre disminuye** al aumentar $K$
- Con $K=1$: inercia máxima (todos los puntos lejos del centroide global)
- Con $K=n$: inercia = 0 (cada punto es su propio cluster)

**Cómo identificar el codo:**
1. Graficar inercia vs $K$
2. Buscar el punto donde la curva cambia de pendiente pronunciada a suave
3. Ese $K$ es un buen balance entre simplicidad y ajuste

**Ventajas:**
- ✅ Muy simple e intuitivo
- ✅ Rápido de calcular

**Limitaciones:**
- ❌ El "codo" no siempre es obvio (curva suave sin punto claro)
- ❌ Subjetivo — dos personas pueden elegir $K$ diferentes
- ❌ Solo aplica a K-Means (usa inercia)

---

### 4.2 Coeficiente de silueta (Silhouette)

**Idea:** Medir qué tan bien está asignado cada punto a su cluster.

**Fórmula para un punto $i$:**

$$
s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}
$$

donde:
- $a(i)$: distancia media de $i$ a los demás puntos de su **mismo** cluster (cohesión)
- $b(i)$: distancia media de $i$ al cluster **vecino más cercano** (separación)

**Rango:** $s(i) \in [-1, 1]$

| Valor | Interpretación |
|-------|----------------|
| **Cercano a 1** | Punto bien asignado — lejos del cluster vecino |
| **Cercano a 0** | Punto en la frontera — podría estar en cualquier cluster |
| **Negativo** | Punto mal asignado — más cerca del cluster vecino |

**Silhouette promedio:**

$$
\bar{s}(K) = \frac{1}{n} \sum_{i=1}^{n} s(i)
$$

**Regla de decisión:** Elegir el $K$ que **maximiza** $\bar{s}(K)$.

**Ventajas:**
- ✅ Fácil de interpretar
- ✅ No requiere conocer las etiquetas verdaderas
- ✅ Funciona con cualquier métrica de distancia

**Limitaciones:**
- ❌ Costoso de calcular para datasets grandes (O(n²))
- ❌ Favorece clusters convexos y compactos

---

### 4.3 Gap Statistic (Tibshirani et al.)

**Idea:** Comparar la inercia observada con la esperada bajo una distribución nula (datos uniformes).

**Fórmula:**

$$
\text{Gap}(K) = \mathbb{E}[\log(W_K^*)] - \log(W_K)
$$

donde:
- $W_K$: inercia con $K$ clusters en los datos reales
- $W_K^*$: inercia con $K$ clusters en datos uniformes (simulados)

**Regla de decisión:** Elegir el $K$ más pequeño tal que:

$$
\text{Gap}(K) \ge \text{Gap}(K+1) - s_{K+1}
$$

donde $s_{K+1}$ es el error estándar de la simulación.

**Ventajas:**
- ✅ Más robusto que el codo
- ✅ Tiene base estadística formal

**Limitaciones:**
- ❌ Más costoso (requiere simulaciones Monte Carlo)
- ❌ Puede ser conservador (elegir $K$ pequeño)

---

### 4.4 Validación de negocio (la más importante)

**Pregunta clave:** ¿Los segmentos son **accionables**?

**Ejemplos:**
- Marketing puede ejecutar solo 3 campañas → $K > 3$ no es útil
- Los segmentos deben tener tamaño suficiente para ser rentables
- Deben ser interpretables (no "cluster 0, cluster 1" sino "clientes leales", "clientes en riesgo")

> 💡 **Recomendación:** Combina **codo + silhouette + validación de negocio**. No confíes en una sola métrica.

In [ ]:
from sklearn.metrics import silhouette_score

# Probamos diferentes valores de K (de 2 a 10)
# No probamos K=1 porque no tiene sentido (un solo cluster = no clustering)
Ks = range(2, 11)

# Listas para guardar las métricas
inercias, siluetas = [], []

# Para cada K, entrenamos K-Means y calculamos métricas
for k in Ks:
    # Entrenamos K-Means con k clusters
    # n_init=10: 10 inicializaciones diferentes, elige la mejor
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    
    # Inercia: suma de distancias al cuadrado de cada punto a su centroide
    # m.inertia_ es un atributo del modelo ya entrenado
    inercias.append(m.inertia_)
    
    # Silhouette: mide cohesión interna vs separación entre clusters
    # silhouette_score calcula el promedio sobre todos los puntos
    # Rango: [-1, 1], mayor es mejor
    siluetas.append(silhouette_score(X, m.labels_))

# --- Visualización ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: Método del codo
axes[0].plot(Ks, inercias, marker='o', linewidth=2, markersize=8, color='blue')
axes[0].set_xlabel('K (número de clusters)')
axes[0].set_ylabel('Inercia (suma de distancias²)')
axes[0].set_title('Método del codo')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Coeficiente de silueta
axes[1].plot(Ks, siluetas, marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[1].set_xlabel('K (número de clusters)')
axes[1].set_ylabel('Silhouette promedio')
axes[1].set_title('Coeficiente de silueta')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Encontramos el K con mayor silhouette
K_optimo = Ks[int(np.argmax(siluetas))]
print(f'K con mayor silueta: K={K_optimo} (score = {max(siluetas):.3f})')

# INTERPRETACIÓN:
# - Codo (izquierda): Busca el punto donde la curva se "dobla"
#   Después de ese punto, agregar más clusters no reduce mucho la inercia
#   En este caso, el codo parece estar en K=4
#
# - Silhouette (derecha): Busca el máximo
#   K=4 tiene el silhouette más alto → clusters bien definidos
#
# CONCLUSIÓN: Ambas métricas sugieren K=4, que coincide con los 4 clusters
# verdaderos que usamos para generar los datos (make_blobs con centers=4)


## 5. Clustering jerárquico

A diferencia de K-Means, **no requiere fijar K** de antemano: produce un árbol completo (**dendrograma**) que muestra cómo se van fusionando los puntos.

### 5.1 Dos estrategias

#### Aglomerativo (bottom-up) — **El más usado**

**Algoritmo:**
1. **Inicio:** Cada punto es su propio cluster ($n$ clusters)
2. **Iteración:** Fusionar los 2 clusters más cercanos
3. **Repetir:** Hasta tener un solo cluster
4. **Resultado:** Dendrograma con toda la jerarquía

**Complejidad:** O(n² log n) — más lento que K-Means pero más informativo.

#### Divisivo (top-down) — Menos común

**Algoritmo:**
1. **Inicio:** Todos los puntos en un solo cluster
2. **Iteración:** Dividir el cluster más heterogéneo
3. **Repetir:** Hasta tener $n$ clusters (cada punto solo)

**Complejidad:** O(2ⁿ) — exponencial, poco práctico.

### 5.2 Linkage — cómo medir distancia entre clusters

La métrica de distancia entre **puntos** es solo la mitad. También necesitamos definir la distancia entre **clusters** (grupos de puntos).

| Linkage | Fórmula | Comportamiento | Cuándo usar |
|---------|---------|----------------|-------------|
| **Single (mínimo)** | $\min_{a \in A, b \in B} d(a, b)$ | Tiende a clusters alargados ("efecto cadena") | Detectar formas irregulares |
| **Complete (máximo)** | $\max_{a \in A, b \in B} d(a, b)$ | Clusters compactos y esféricos | Clusters bien separados |
| **Average** | $\frac{1}{|A||B|} \sum_{a \in A, b \in B} d(a, b)$ | Balance entre single y complete | Uso general |
| **Ward** | Minimiza incremento de $\sum \|x_i - \mu_k\|^2$ | Similar a K-Means, clusters de tamaño parecido | **Default recomendado** |

**Ejemplo visual:**

Imagina dos clusters A y B:
```
A: ●  ●  ●
         
B:       ● ● ●
```

- **Single:** Distancia entre los 2 puntos más cercanos (uno de A, uno de B)
- **Complete:** Distancia entre los 2 puntos más lejanos
- **Average:** Promedio de todas las distancias entre pares
- **Ward:** Cuánto aumentaría la inercia total si fusionamos A y B

> 💡 **Recomendación:** Usa **Ward** cuando los datos están escalados y esperas clusters de tamaño similar. Es el más robusto en la práctica.

### 5.3 Ventajas y limitaciones

**Ventajas:**
- ✅ No necesitas fijar $K$ de antemano
- ✅ Produce un dendrograma que muestra la estructura completa
- ✅ Acepta cualquier métrica de distancia
- ✅ Determinístico (no depende de inicialización aleatoria)
- ✅ Útil para exploración de datos

**Limitaciones:**
- ❌ Más lento que K-Means: O(n² log n) vs O(nKi)
- ❌ No escala bien a millones de observaciones
- ❌ No tiene `predict()` — para asignar nuevos puntos hay que reajustar
- ❌ Sensible a outliers (especialmente con single linkage)
- ❌ Una vez fusionados dos clusters, no se pueden separar (decisión irreversible)

**Cuándo usar:**
- No sabes cuántos clusters esperar
- Quieres explorar la estructura jerárquica
- Tienes pocos datos (< 10k observaciones)
- Necesitas usar una métrica de distancia específica (no euclidiana)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Submuestra para que el dendrograma sea legible
# Con 400 puntos el dendrograma sería ilegible (muchas hojas)
# Tomamos 80 puntos aleatorios
idx = np.random.default_rng(0).choice(len(X), size=80, replace=False)
X_sub = X[idx]

# Calculamos la matriz de linkage (fusiones jerárquicas)
# method='ward': minimiza la varianza intra-cluster (similar a K-Means)
# Z es una matriz (n-1 × 4) donde cada fila es una fusión:
#   [cluster_1, cluster_2, distancia, n_observaciones]
Z = linkage(X_sub, method='ward')

# --- Visualización del dendrograma ---
fig, ax = plt.subplots(figsize=(11, 5))

# dendrogram dibuja el árbol jerárquico
# color_threshold=12: fusiones por debajo de esta altura tienen el mismo color
#                     fusiones por encima tienen colores diferentes (clusters)
dendrogram(Z, ax=ax, color_threshold=12)

# Línea horizontal roja: corte sugerido para obtener K clusters
# La altura del corte determina cuántos clusters obtenemos
ax.axhline(12, color='red', linestyle='--', linewidth=2, label='Corte sugerido (K=4)')

ax.set_title('Dendrograma — Clustering Jerárquico Aglomerativo (linkage=Ward)')
ax.set_xlabel('Índice de observación')
ax.set_ylabel('Distancia de fusión (altura)')
ax.legend()
plt.show()

# Extraemos los clusters cortando el dendrograma a una altura específica
# fcluster asigna cada punto a un cluster según el corte
# t=4: queremos 4 clusters
# criterion='maxclust': cortar para obtener exactamente 4 clusters
labels_h = fcluster(Z, t=4, criterion='maxclust')

print(f'Clusters obtenidos al cortar en K=4: {np.unique(labels_h)}')
print(f'Tamaño de cada cluster: {np.bincount(labels_h)[1:]}')  # [1:] para ignorar el índice 0

# CÓMO LEER EL DENDROGRAMA:
# - Eje X: Observaciones individuales (hojas del árbol)
# - Eje Y: Distancia (altura) a la que se fusionaron
# - Ramas del mismo color: pertenecen al mismo cluster (según el corte)
# - Ramas que se fusionan TEMPRANO (abajo): observaciones muy similares
# - Ramas que se fusionan TARDE (arriba): clusters bien separados
#
# CÓMO ELEGIR K:
# 1. Buscar el MAYOR SALTO VERTICAL sin cruce horizontal
# 2. Trazar una línea horizontal a esa altura
# 3. El número de ramas que cruza = número de clusters
#
# En este caso, el mayor salto está alrededor de altura 12-15
# La línea roja cruza 4 ramas → K=4 clusters


## 6. Caso práctico — Telco Customer Churn

**Dataset:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn
**Archivo:** `WA_Fn-UseC_-Telco-Customer-Churn.csv` (7.043 × 21).

En este módulo **no estamos prediciendo** `Churn`. Lo que queremos es descubrir **segmentos naturales de clientes** a partir de su comportamiento de uso y facturación. Para mantener el ejemplo interpretable usamos solo variables numéricas:

- `tenure` — meses como cliente.
- `MonthlyCharges` — cuota mensual (USD).
- `TotalCharges` — facturación acumulada (USD).

Al final usaremos `Churn` solo como **validación externa** para ver si los segmentos descubiertos tienen sentido.

In [ ]:
from pathlib import Path
import pandas as pd

# Definimos la ruta al CSV usando pathlib (más robusto que strings de ruta)
DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Verificamos que el archivo existe antes de intentar cargarlo
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/blastchar/telco-customer-churn '
        'y colócalo en data/ (ver README.md).'
    )

# Cargamos el CSV completo en un DataFrame de pandas
# Este dataset tiene 7043 filas (clientes) y 21 columnas
df = pd.read_csv(DATA)

# PROBLEMA CON TotalCharges: viene como string con espacios en blanco para algunos clientes
# pd.to_numeric convierte a numérico, errors='coerce' pone NaN donde no puede convertir
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Eliminamos filas con TotalCharges faltante (son muy pocas)
# reset_index(drop=True) reinicia el índice después de eliminar filas
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)

print('Shape:', df.shape)
print(f'Clientes después de limpiar: {len(df)}')

# Mostramos las primeras 5 filas para inspeccionar la estructura
df.head()

# CONTEXTO DEL DATASET:
# - Cada fila es un cliente de una compañía de telecomunicaciones
# - Variables demográficas: gender, SeniorCitizen, Partner, Dependents
# - Servicios contratados: PhoneService, InternetService, StreamingTV, etc.
# - Información del contrato: Contract, PaymentMethod, PaperlessBilling
# - Facturación: MonthlyCharges, TotalCharges, tenure (meses como cliente)
# - Target (que NO usaremos para clustering): Churn (si se dio de baja)


In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Configuramos el estilo visual para todos los gráficos
sns.set_theme(style='whitegrid')

# ─────────────────────────────────────────────────────────────────
# Selección de features
# ─────────────────────────────────────────────────────────────────
# Para mantener el ejemplo interpretable, usamos solo 3 variables numéricas
# En un caso real, podrías usar más features (incluyendo categóricas codificadas)

features = ['tenure',         # meses como cliente (0-72)
            'MonthlyCharges',  # cuota mensual en USD (18-118)
            'TotalCharges']    # facturación acumulada en USD (18-8684)

# Extraemos solo las features seleccionadas
X_raw = df[features].copy()

# ─────────────────────────────────────────────────────────────────
# Estandarización (CRÍTICO para K-Means)
# ─────────────────────────────────────────────────────────────────
# StandardScaler transforma cada variable a media=0, std=1
# Esto es ESENCIAL porque:
# 1. TotalCharges tiene rango [18, 8684] — dominaría la distancia euclidiana
# 2. tenure tiene rango [0, 72] — sería casi ignorado sin escalar
# 3. K-Means usa distancia euclidiana → sensible a la escala

# fit() calcula media y std de cada columna
scaler = StandardScaler().fit(X_raw)

# transform() aplica la transformación: (x - media) / std
X_s = scaler.transform(X_raw)

print('Variables escaladas: media≈0, std≈1')
print('Verificación:')
pd.DataFrame(X_s, columns=features).describe().round(2).loc[['mean', 'std']]

# RESULTADO ESPERADO:
# - mean ≈ 0 (o muy cercano, puede haber pequeños errores numéricos)
# - std ≈ 1
#
# Ahora las 3 variables tienen la misma "importancia" en el cálculo de distancias


In [ ]:
# --- Elegir K con codo + silueta ---

# Probamos diferentes valores de K (de 2 a 8)
Ks = range(2, 9)

# Listas para guardar las métricas
inercias, siluetas = [], []

# Para cada K, entrenamos K-Means y calculamos métricas
for k in Ks:
    # Entrenamos K-Means con k clusters
    # n_init=10: 10 inicializaciones diferentes, elige la mejor por inercia
    # random_state=42: semilla fija para reproducibilidad
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_s)
    
    # Inercia: suma de distancias al cuadrado (menor = clusters más compactos)
    inercias.append(m.inertia_)
    
    # Silhouette: cohesión interna vs separación (mayor = mejor)
    # Rango: [-1, 1], valores cercanos a 1 indican clusters bien definidos
    siluetas.append(silhouette_score(X_s, m.labels_))

# --- Visualización ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: Método del codo
axes[0].plot(Ks, inercias, marker='o', linewidth=2, markersize=8)
axes[0].set_title('Método del Codo')
axes[0].set_xlabel('K (número de clusters)')
axes[0].set_ylabel('Inercia')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Coeficiente de silueta
axes[1].plot(Ks, siluetas, marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[1].set_title('Coeficiente de Silueta')
axes[1].set_xlabel('K (número de clusters)')
axes[1].set_ylabel('Silhouette promedio')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Encontramos el K con mayor silhouette
K = Ks[int(np.argmax(siluetas))]
print(f'K elegido (max silueta): {K} (score = {max(siluetas):.3f})')

# INTERPRETACIÓN:
# - Codo: Busca donde la curva se aplana (mejora marginal pequeña)
# - Silhouette: Busca el máximo (mejor balance cohesión/separación)
#
# IMPORTANTE: En datos reales, codo y silhouette pueden sugerir K diferentes
# En ese caso, considera:
# 1. ¿Qué K tiene sentido de negocio? (e.g., marketing puede ejecutar 3 campañas)
# 2. ¿Los segmentos son interpretables?
# 3. ¿Tienen tamaño suficiente para ser accionables?


In [ ]:
# --- K-Means final con el K elegido ---

# Entrenamos K-Means con el K óptimo (según silhouette)
# n_init=20: más inicializaciones para asegurar convergencia al mejor mínimo
km = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_s)

# Asignamos el cluster a cada cliente en el DataFrame original
# km.labels_: array con el cluster asignado a cada observación (0, 1, 2, ...)
df['cluster_km'] = km.labels_

# ─────────────────────────────────────────────────────────────────
# Análisis de centroides (en escala original)
# ─────────────────────────────────────────────────────────────────
# Los centroides están en escala estandarizada (media=0, std=1)
# Para interpretarlos, los devolvemos a la escala original

# inverse_transform deshace la estandarización: x_original = x_scaled * std + media
centros = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_),
    columns=features,
).round(1)

# Agregamos el tamaño de cada cluster
centros.index.name = 'cluster'
centros['n_clientes'] = df['cluster_km'].value_counts().sort_index().values

print('Centroides de cada cluster (escala original):')
centros

# CÓMO INTERPRETAR LOS CENTROIDES:
# Cada fila es el "cliente promedio" de ese cluster
# 
# Ejemplo de interpretación (adaptar a tus resultados):
# - Cluster 0: tenure bajo, charges bajos → "Clientes nuevos"
# - Cluster 1: tenure alto, charges altos → "Clientes leales de alto valor"
# - Cluster 2: tenure bajo, MonthlyCharges alto → "Clientes premium recientes"
#
# Estos nombres son hipótesis que debes validar con el negocio
# y con análisis adicionales (e.g., tasa de churn por cluster)


In [ ]:
# --- Clustering jerárquico (Ward) sobre una submuestra para comparar ---

# Tomamos una submuestra de 1000 clientes
# El clustering jerárquico es O(n² log n) → muy lento para 7000+ observaciones
# En producción, si necesitas jerárquico con muchos datos, considera BIRCH o mini-batch
sub = df.sample(n=1000, random_state=42)

# Estandarizamos la submuestra (usando el mismo scaler del train completo)
X_sub = scaler.transform(sub[features])

# Aplicamos clustering jerárquico aglomerativo
# n_clusters=K: cortamos el dendrograma para obtener K clusters
# linkage='ward': minimiza la varianza intra-cluster (similar a K-Means)
agg = AgglomerativeClustering(n_clusters=K, linkage='ward').fit(X_sub)

# Asignamos los clusters del jerárquico
sub['cluster_h'] = agg.labels_

# También asignamos los clusters de K-Means para comparar
# km.predict asigna cada punto al centroide más cercano
sub['cluster_km'] = km.predict(X_sub)

# ─────────────────────────────────────────────────────────────────
# Comparamos las asignaciones de ambos métodos
# ─────────────────────────────────────────────────────────────────
# Tabla de contingencia: filas = K-Means, columnas = Jerárquico
ct = pd.crosstab(sub['cluster_km'], sub['cluster_h'],
                 rownames=['K-Means'], colnames=['Jerárquico'])

print('Concordancia entre los dos métodos sobre la submuestra:')
print(ct)

# INTERPRETACIÓN:
# - Diagonal principal: clientes asignados al mismo cluster por ambos métodos
# - Fuera de la diagonal: desacuerdos
#
# Si hay alta concordancia (diagonal dominante), ambos métodos encuentran
# la misma estructura → los clusters son robustos
#
# Si hay baja concordancia, puede significar:
# 1. Los clusters no están bien definidos (estructura débil)
# 2. Los métodos capturan aspectos diferentes de los datos
# 3. El linkage elegido no es apropiado
#
# NOTA: Los números de cluster (0, 1, 2) son arbitrarios
# El cluster 0 de K-Means NO necesariamente corresponde al cluster 0 del jerárquico


In [ ]:
# --- Visualización: pares de variables coloreados por cluster ---

# Creamos 3 subplots para visualizar las 3 combinaciones posibles de pares
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Definimos los 3 pares de variables a graficar
pares = [('tenure', 'MonthlyCharges'),
         ('tenure', 'TotalCharges'),
         ('MonthlyCharges', 'TotalCharges')]

# Para cada par, creamos un scatter plot coloreado por cluster
for ax, (a, b) in zip(axes, pares):
    # sns.scatterplot colorea automáticamente según 'hue'
    # palette='tab10': paleta de colores categóricos
    # s=15: tamaño de los puntos
    # alpha=0.6: transparencia para ver superposiciones
    # legend=False: no mostramos leyenda (ocupa mucho espacio)
    sns.scatterplot(x=a, y=b, hue='cluster_km', data=df,
                    palette='tab10', s=15, alpha=0.6, ax=ax, legend=False)
    ax.set_title(f'{a} vs {b}')
    ax.set_xlabel(a)
    ax.set_ylabel(b)

plt.tight_layout()
plt.show()

# CÓMO INTERPRETAR:
# - Busca SEPARACIÓN entre colores → clusters bien definidos
# - Si los colores se entremezclan mucho → clusters débiles en esas dimensiones
# - Busca PATRONES:
#   * Grupos en esquinas → segmentos extremos
#   * Grupos lineales → correlación entre variables dentro del cluster
#   * Grupos compactos → clusters homogéneos
#
# EJEMPLO DE INTERPRETACIÓN (adaptar a tus datos):
# - tenure vs MonthlyCharges: ¿Hay un grupo de clientes nuevos (tenure bajo)
#   con charges altos (premium)?
# - tenure vs TotalCharges: Correlación positiva esperada (más meses → más gasto)
#   pero ¿hay clusters con pendientes diferentes?
# - MonthlyCharges vs TotalCharges: ¿Hay clientes con gasto mensual alto
#   pero total bajo (recién llegados premium)?


### Interpretación de los segmentos

A partir de los centroides en escala original, podemos darle un nombre comercial a cada cluster (los nombres dependen del K que haya salido — adapta la lectura a tus centroides):

- **Clientes nuevos / bajo gasto**: `tenure` corto, `TotalCharges` bajo.
- **Clientes leales / alto valor**: `tenure` alto, `TotalCharges` alto.
- **Clientes premium recientes**: `MonthlyCharges` alto pero `tenure` corto — son los más riesgosos: pagan caro y aún no se "atan" a la compañía.

Esto es lo que vuelve útil el clustering: **no es predecir** sino **describir** la base de clientes para diseñar acciones diferenciadas.

In [ ]:
# --- Validación externa con el target real (que NO usamos para entrenar) ---

# Convertimos la variable Churn (categórica) a binaria (0/1)
# 'Yes' → 1 (se dio de baja), 'No' → 0 (se quedó)
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)

# Calculamos la tasa de churn por cluster
# groupby agrupa por cluster, agg calcula estadísticas
churn_por_cluster = df.groupby('cluster_km').agg(
    n_clientes=('Churn_bin', 'size'),      # tamaño del cluster
    tasa_churn=('Churn_bin', 'mean'),      # proporción de churn (0 a 1)
).round(3)

print('Tasa de churn por cluster:')
churn_por_cluster

# INTERPRETACIÓN CLAVE:
# Si la tasa de churn VARÍA MUCHO entre clusters, significa que los segmentos
# descubiertos están capturando algo REAL del negocio, aunque NUNCA usamos
# la variable Churn para entrenar el clustering.
#
# Esto es la MEJOR VALIDACIÓN posible para un problema no supervisado:
# el patrón se sostiene contra una etiqueta independiente.
#
# EJEMPLO DE INTERPRETACIÓN (adaptar a tus resultados):
# - Cluster 0: tasa_churn = 0.45 (45%) → "Clientes en riesgo"
# - Cluster 1: tasa_churn = 0.10 (10%) → "Clientes leales"
# - Cluster 2: tasa_churn = 0.30 (30%) → "Clientes premium pero volátiles"
#
# ACCIÓN DE NEGOCIO:
# - Cluster de alto churn → campaña de retención agresiva
# - Cluster de bajo churn → programa de fidelización, upselling
# - Cluster premium volátil → mejorar experiencia, atención personalizada


Si la **tasa de churn varía mucho entre clusters**, los segmentos detectados están capturando algo real del negocio aunque nunca usamos `Churn` para entrenar. Esto es la mejor validación posible para un problema no supervisado: el patrón se sostiene contra una etiqueta independiente.

## 7. Referencias

- ISLR cap. 12.4 — *Clustering Methods*. https://www.statlearning.com/
- Lloyd, S. (1982). *Least squares quantization in PCM* (algoritmo K-Means).
- Ward, J. H. (1963). *Hierarchical grouping to optimize an objective function*.
- Rousseeuw, P. (1987). *Silhouettes: A graphical aid to the interpretation and validation of cluster analysis*.
- Tibshirani, R., Walther, G. & Hastie, T. (2001). *Estimating the number of clusters in a data set via the gap statistic*.
- scikit-learn: https://scikit-learn.org/stable/modules/clustering.html
- Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

## Predicción sobre datos nuevos — uso del modelo en producción

Aunque "predecir" en clustering significa cosas distintas según el algoritmo, sí podemos persistir un modelo y asignar nuevos clientes a los segmentos descubiertos:

1. **Reentrenar con todos los datos** (ya validamos K, ahora aprovechamos toda la información).
2. **Persistir el `scaler` junto al modelo** — sin la misma escala, las distancias no significan lo mismo.
3. **Asignar** cada nuevo cliente al cluster del centroide más cercano con `kmeans.predict(...)`.

> Limitación: K-Means tiene `predict`, pero **el clustering jerárquico no** — habría que reajustarlo con el nuevo punto incluido o entrenar un clasificador supervisado sobre los labels.

In [ ]:
import joblib

# ─────────────────────────────────────────────────────────────────
# PASO 1: Reentrenamos con TODOS los datos disponibles
# ─────────────────────────────────────────────────────────────────
# Ya validamos K con codo/silhouette. Ahora usamos el 100% de los datos
# para que el modelo final tenga los mejores centroides posibles.

# Reajustamos el scaler con todos los datos
scaler_final = StandardScaler().fit(df[features])

# Entrenamos K-Means final con todos los datos estandarizados
modelo_final = KMeans(n_clusters=K, n_init=20, random_state=42).fit(
    scaler_final.transform(df[features])
)

# ─────────────────────────────────────────────────────────────────
# PASO 2: Creamos un bundle con todo lo necesario para producción
# ─────────────────────────────────────────────────────────────────
# CRÍTICO: No basta con guardar el modelo
# También necesitamos el scaler y la lista de features

bundle = {
    'modelo':   modelo_final,      # K-Means entrenado
    'scaler':   scaler_final,      # StandardScaler (media y std de cada feature)
    'features': features,          # Lista de features en el orden correcto
    'K':        K,                 # Número de clusters (para documentación)
}

# ─────────────────────────────────────────────────────────────────
# PASO 3: Persistimos el bundle a disco
# ─────────────────────────────────────────────────────────────────
joblib.dump(bundle, 'modelo_telco_kmeans.pkl')
print('Bundle guardado: modelo_telco_kmeans.pkl')

# Para recuperarlo en otro proceso (API, script batch, notebook diferente):
# bundle = joblib.load('modelo_telco_kmeans.pkl')
# modelo = bundle['modelo']
# scaler = bundle['scaler']
# features = bundle['features']

# IMPORTANTE: En producción real también deberías guardar:
# - Fecha de entrenamiento y versión del modelo
# - Métricas de validación (silhouette, inercia)
# - Descripción de cada cluster (nombres, características)
# - Distribuciones de las features (para detectar data drift)
# - Tamaño de cada cluster (para monitorear cambios en la composición)


### Inferencia individual — un cliente nuevo

In [ ]:
# Definimos un cliente hipotético
# En producción esto vendría de una API (JSON), base de datos, CRM, etc.
nuevo_cliente = pd.DataFrame([{
    'tenure':         3,      # 3 meses como cliente (nuevo)
    'MonthlyCharges': 95.0,   # $95/mes (alto)
    'TotalCharges':   285.0,  # $285 total (3 meses × $95)
}])

# CRÍTICO: Aplicar la MISMA transformación que en el entrenamiento
# 1. Estandarizar con el scaler_final (usa la media/std del train completo)
nuevo_scaled = scaler_final.transform(nuevo_cliente)

# 2. Predecir el cluster asignando al centroide más cercano
# predict() calcula la distancia euclidiana a cada centroide y elige el mínimo
cluster_id = modelo_final.predict(nuevo_scaled)[0]

print(f'El cliente cae en el cluster: {cluster_id}')
print('\nCentroide de ese cluster (escala original):')
print(centros.loc[cluster_id])

# INTERPRETACIÓN:
# - El cliente tiene tenure bajo (3 meses) pero MonthlyCharges alto ($95)
# - Probablemente caiga en un cluster de "Clientes premium recientes"
# - Estos clientes son valiosos pero tienen alto riesgo de churn
#   (no se han "atado" a la compañía todavía)
#
# ACCIÓN DE NEGOCIO:
# - Ofrecerle un descuento por contratar plan anual (reducir churn)
# - Programa de bienvenida personalizado
# - Monitorear satisfacción de cerca (encuestas, NPS)
# - Cross-selling de servicios adicionales (aumentar switching costs)


**Lecciones para producción:**
- El `K` óptimo depende de qué decisiones de negocio quieres soportar — una segmentación con K=4 puede ser estadísticamente buena pero inútil si marketing solo puede ejecutar 2 campañas.
- Refrescar los clusters cada cierto tiempo: el comportamiento de los clientes cambia (data drift). Un cluster que existía el año pasado puede haberse difuminado.
- Documentar **qué significa cada cluster** en lenguaje de negocio, no como "cluster 0, cluster 1".